# Ingesting dimension data into bronze layer
Reading raw CSVs, adding metadata columns, writing as delta tables.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F
catalog_name = 'edutrack'

## Students

In [0]:
student_schema = StructType([
    StructField('student_id',    StringType(),  False),
    StructField('name',          StringType(),  True),
    StructField('department',    StringType(),  True),
    StructField('year_of_study', IntegerType(), True),
    StructField('date_of_birth', StringType(),  True),
    StructField('phone',         StringType(),  True),
    StructField('city',          StringType(),  True),
    StructField('email',         StringType(),  True),
])
df = (spark.read.option('header','true').option('delimiter',',')
      .schema(student_schema)
      .csv('/Volumes/edutrack/source_data/raw/students/*.csv')
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .withColumn('_ingested_at', F.current_timestamp()))
display(df.limit(5))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.bronze.brz_students')
print('brz_students:', spark.table('edutrack.bronze.brz_students').count(),'rows')

## Courses

In [0]:
course_schema = StructType([
    StructField('course_id',   StringType(),  False),
    StructField('course_name', StringType(),  True),
    StructField('department',  StringType(),  True),
    StructField('credits',     IntegerType(), True),
    StructField('level',       StringType(),  True),
])
df = (spark.read.option('header','true').option('delimiter',',')
      .schema(course_schema)
      .csv('/Volumes/edutrack/source_data/raw/courses/*.csv')
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .withColumn('_ingested_at', F.current_timestamp()))
display(df.limit(5))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.bronze.brz_courses')
print('brz_courses:', spark.table('edutrack.bronze.brz_courses').count(),'rows')

## Instructors

In [0]:
instructor_schema = StructType([
    StructField('instructor_id',  StringType(),  False),
    StructField('name',           StringType(),  True),
    StructField('department',     StringType(),  True),
    StructField('experience_yrs', IntegerType(), True),
    StructField('email',          StringType(),  True),
])
df = (spark.read.option('header','true').option('delimiter',',')
      .schema(instructor_schema)
      .csv('/Volumes/edutrack/source_data/raw/instructors/*.csv')
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .withColumn('_ingested_at', F.current_timestamp()))
display(df.limit(5))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.bronze.brz_instructors')
print('brz_instructors:', spark.table('edutrack.bronze.brz_instructors').count(),'rows')

## Date Dimension

In [0]:
date_schema = StructType([
    StructField('date',         StringType(),  True),
    StructField('year',         IntegerType(), True),
    StructField('month',        IntegerType(), True),
    StructField('month_name',   StringType(),  True),
    StructField('day_name',     StringType(),  True),
    StructField('quarter',      IntegerType(), True),
    StructField('week_of_year', IntegerType(), True),
    StructField('is_weekend',   IntegerType(), True),
])
df = (spark.read.option('header','true').option('delimiter',',')
      .schema(date_schema)
      .csv('/Volumes/edutrack/source_data/raw/date/*.csv')
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .withColumn('_ingested_at', F.current_timestamp()))
display(df.limit(5))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.bronze.brz_date')
print('brz_date:', spark.table('edutrack.bronze.brz_date').count(),'rows')

## Sanity Check

In [0]:
for t in ['brz_students','brz_courses','brz_instructors','brz_date']:
    cnt = spark.sql(f'SELECT count(*) as cnt FROM edutrack.bronze.{t}').collect()[0]['cnt']
    print(f'{t:30s}  {cnt:>6} rows')